# EDA: pick-and-roll coverage recognition (analytics/coverage)

Tables only, from `analytics.coverage.eda`. No hand labels exist yet (STATUS.md): this notebook reads the synthetic fixture season, so every number it shows is fixture output, not a result. Held-out games are set aside before anything is computed; nothing here tunes a model.

Run from the repository root (or from `notebooks/`): the first cell finds the root.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "analytics").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from analytics.coverage import eda
from analytics.coverage.features import feature_columns

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
feats = eda.fixture_feature_frame(seed=0, n_games=8, heldout_every=4)
print("fixture output, not a result:", feats.shape, "rows x columns;", len(feature_columns()), "features")

## Class balance (labelled rows of training games)

In [ ]:
eda.class_balance(feats)

## Feature distributions by class

Mean, standard deviation, minimum and maximum of every feature per class. The features that should move with the class: `d2_nearest_end` (switch), `d2_basket_dist_min_ft` and `d2_depth_end_ft` (drop), `d2_handler_dist_min_post_ft` with `d2_handler_dist_after_min_ft` (hedge recovers, blitz stays), `handler_lat_end_ft` and `d1_lat_rel_handler_end_ft` (ice), `d1_depth_rel_screener_pass_ft` (over negative, under positive), `both_within_6ft_frac` (trap).

In [ ]:
dist = eda.feature_distributions_by_class(feats)
dist.pivot(index="feature", columns="class", values="mean").round(2)

In [ ]:
eda.class_separation(feats).round(3)

## Correlation table

Pearson correlation of the features; pairs above 0.9 in absolute value are listed so a redundant pair is visible.

In [ ]:
import numpy as np

corr = eda.correlation_table(feats)
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
pairs[pairs.abs() > 0.9].sort_values(key=abs, ascending=False).to_frame("r")

In [ ]:
corr.round(2)

## Shuffled-label check

Out-of-fold macro F1 of the logistic candidate on the true labels and on labels permuted across rows, next to the majority baseline. The permuted row must sit near or below majority: a model that scores on permuted labels is reading something it should not.

In [ ]:
eda.shuffled_label_check(feats, seed=0, n_folds=5).round(3)

Fixture output, not a result. The real run needs the hand-label tables of lane ACTIONS under `DATA_ROOT/labels/` on the owner's Mac: `python -m analytics.coverage.train --data "<Drive>/nba_harvest" --out "<Drive>/nba_harvest" --report reports/coverage.md --folds 5 --seed 0 --bootstrap 1000`.